# Anomaly Scores Visualization

This notebook visualizes both normalized and non-normalized anomaly scores for both panels.

## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path
import re

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pio.templates.default = 'plotly_white'
%matplotlib inline

# Paths
DATA_DIR = Path('data')
RESULTS_BASE_DIR = Path('..') / DATA_DIR / 'results'

# Configuration - Set to None to use latest
RESULTS_TIMESTAMP = None  # e.g., '20260126_154153' or None for latest
MODEL_NAME = 'LightGBM'

print('Environment configured successfully!')

## 2. Load Data

In [ ]:
def find_latest_timestamped_dir(base_dir):
    """Find the most recent timestamped directory."""
    if not base_dir.exists():
        raise FileNotFoundError(f'Directory not found: {base_dir}')
    
    timestamp_pattern = re.compile(r'^\d{8}_\d{6}$')
    timestamped_dirs = [
        d for d in base_dir.iterdir() 
        if d.is_dir() and timestamp_pattern.match(d.name)
    ]
    
    if not timestamped_dirs:
        raise FileNotFoundError(f'No timestamped directories found in: {base_dir}')
    
    return sorted(timestamped_dirs, key=lambda d: d.name, reverse=True)[0]

# Get results directory
results_dir = (
    RESULTS_BASE_DIR / RESULTS_TIMESTAMP if RESULTS_TIMESTAMP 
    else find_latest_timestamped_dir(RESULTS_BASE_DIR)
)
print(f'Using results from: {results_dir.name}')

In [ ]:
# Load scores for both panels
df_scores_p1 = pd.read_csv(results_dir / f'{MODEL_NAME}_panel_1_scores.csv')
df_scores_p2 = pd.read_csv(results_dir / f'{MODEL_NAME}_panel_2_scores.csv')

# Load flags for both panels
df_flags_p1 = pd.read_csv(results_dir / f'{MODEL_NAME}_panel_1_flags.csv')
df_flags_p2 = pd.read_csv(results_dir / f'{MODEL_NAME}_panel_2_flags.csv')

# Convert timestamps
for df in [df_scores_p1, df_scores_p2, df_flags_p1, df_flags_p2]:
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df.set_index('timestamp', inplace=True)

# Convert numeric columns
for col in df_scores_p1.columns:
    df_scores_p1[col] = pd.to_numeric(df_scores_p1[col], errors='coerce')
    
for col in df_scores_p2.columns:
    df_scores_p2[col] = pd.to_numeric(df_scores_p2[col], errors='coerce')

# Convert flags to numeric
for col in df_flags_p1.columns:
    df_flags_p1[col] = pd.to_numeric(df_flags_p1[col], errors='coerce').fillna(0).astype(int)
    
for col in df_flags_p2.columns:
    df_flags_p2[col] = pd.to_numeric(df_flags_p2[col], errors='coerce').fillna(0).astype(int)

# Merge flags with scores for easier plotting
df_scores_p1['anomaly_flag'] = df_flags_p1['anomaly_flag'] if 'anomaly_flag' in df_flags_p1.columns else 0
df_scores_p2['anomaly_flag'] = df_flags_p2['anomaly_flag'] if 'anomaly_flag' in df_flags_p2.columns else 0

print(f'Panel 1 scores shape: {df_scores_p1.shape}')
print(f'Panel 1 columns: {list(df_scores_p1.columns)}')
print(f'\nPanel 2 scores shape: {df_scores_p2.shape}')
print(f'Panel 2 columns: {list(df_scores_p2.columns)}')

# Display summary statistics
print('\n=== Panel 1 Score Statistics ===')
print(df_scores_p1.describe())
print(f'\nPanel 1 anomalies detected: {df_scores_p1["anomaly_flag"].sum()}')
print('\n=== Panel 2 Score Statistics ===')
print(df_scores_p2.describe())
print(f'\nPanel 2 anomalies detected: {df_scores_p2["anomaly_flag"].sum()}')

## 3. Plot Scores - Matplotlib

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Anomaly Scores Comparison: Normalized vs Non-Normalized', fontsize=16, fontweight='bold')

# Panel 1 - Non-normalized scores
ax1 = axes[0, 0]
ax1.plot(df_scores_p1.index, df_scores_p1['anomaly_score'], alpha=0.7, linewidth=1, color='#1f77b4', label='Score')
# Add anomaly markers
anomaly_mask_p1 = df_scores_p1['anomaly_flag'] == 1
if anomaly_mask_p1.any():
    ax1.scatter(df_scores_p1.index[anomaly_mask_p1], df_scores_p1['anomaly_score'][anomaly_mask_p1], 
                color='red', s=30, marker='o', alpha=0.7, zorder=5, label='Anomaly')
ax1.set_title('Panel 1: Non-Normalized Scores', fontsize=12, fontweight='bold')
ax1.set_xlabel('Timestamp')
ax1.set_ylabel('Anomaly Score')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)
ax1.legend()

# Panel 1 - Normalized scores
ax2 = axes[0, 1]
ax2.plot(df_scores_p1.index, df_scores_p1['anomaly_score_normalized'], alpha=0.7, linewidth=1, color='#ff7f0e', label='Score')
# Add anomaly markers
if anomaly_mask_p1.any():
    ax2.scatter(df_scores_p1.index[anomaly_mask_p1], df_scores_p1['anomaly_score_normalized'][anomaly_mask_p1], 
                color='red', s=30, marker='o', alpha=0.7, zorder=5, label='Anomaly')
ax2.set_title('Panel 1: Normalized Scores [0, 1]', fontsize=12, fontweight='bold')
ax2.set_xlabel('Timestamp')
ax2.set_ylabel('Normalized Anomaly Score')
ax2.set_ylim([0, 1])
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)
ax2.legend()

# Panel 2 - Non-normalized scores
ax3 = axes[1, 0]
ax3.plot(df_scores_p2.index, df_scores_p2['anomaly_score'], alpha=0.7, linewidth=1, color='#2ca02c', label='Score')
# Add anomaly markers
anomaly_mask_p2 = df_scores_p2['anomaly_flag'] == 1
if anomaly_mask_p2.any():
    ax3.scatter(df_scores_p2.index[anomaly_mask_p2], df_scores_p2['anomaly_score'][anomaly_mask_p2], 
                color='red', s=30, marker='o', alpha=0.7, zorder=5, label='Anomaly')
ax3.set_title('Panel 2: Non-Normalized Scores', fontsize=12, fontweight='bold')
ax3.set_xlabel('Timestamp')
ax3.set_ylabel('Anomaly Score')
ax3.grid(True, alpha=0.3)
ax3.tick_params(axis='x', rotation=45)
ax3.legend()

# Panel 2 - Normalized scores
ax4 = axes[1, 1]
ax4.plot(df_scores_p2.index, df_scores_p2['anomaly_score_normalized'], alpha=0.7, linewidth=1, color='#d62728', label='Score')
# Add anomaly markers
if anomaly_mask_p2.any():
    ax4.scatter(df_scores_p2.index[anomaly_mask_p2], df_scores_p2['anomaly_score_normalized'][anomaly_mask_p2], 
                color='red', s=30, marker='o', alpha=0.7, zorder=5, label='Anomaly')
ax4.set_title('Panel 2: Normalized Scores [0, 1]', fontsize=12, fontweight='bold')
ax4.set_xlabel('Timestamp')
ax4.set_ylabel('Normalized Anomaly Score')
ax4.set_ylim([0, 1])
ax4.grid(True, alpha=0.3)
ax4.tick_params(axis='x', rotation=45)
ax4.legend()

plt.tight_layout()
plt.show()

## 4. Plot Scores - Side by Side Comparison

In [ ]:
# Create figure comparing normalized vs non-normalized for each panel
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Anomaly Scores: Normalized vs Non-Normalized Comparison', fontsize=16, fontweight='bold')

# Panel 1 - Both scores on same plot
ax1 = axes[0]
ax1_twin = ax1.twinx()

line1 = ax1.plot(df_scores_p1.index, df_scores_p1['anomaly_score'], 
                 alpha=0.7, linewidth=1.5, color='#1f77b4', label='Non-Normalized')
line2 = ax1_twin.plot(df_scores_p1.index, df_scores_p1['anomaly_score_normalized'], 
                      alpha=0.7, linewidth=1.5, color='#ff7f0e', label='Normalized')

# Add anomaly markers
anomaly_mask_p1 = df_scores_p1['anomaly_flag'] == 1
if anomaly_mask_p1.any():
    marker1 = ax1.scatter(df_scores_p1.index[anomaly_mask_p1], df_scores_p1['anomaly_score'][anomaly_mask_p1], 
                          color='red', s=40, marker='o', alpha=0.8, zorder=5, label='Anomaly')
    marker2 = ax1_twin.scatter(df_scores_p1.index[anomaly_mask_p1], df_scores_p1['anomaly_score_normalized'][anomaly_mask_p1], 
                                color='red', s=40, marker='o', alpha=0.8, zorder=5)

ax1.set_title('Panel 1: Normalized vs Non-Normalized Scores', fontsize=12, fontweight='bold')
ax1.set_xlabel('Timestamp')
ax1.set_ylabel('Anomaly Score (Non-Normalized)', color='#1f77b4')
ax1_twin.set_ylabel('Anomaly Score (Normalized)', color='#ff7f0e')
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1_twin.tick_params(axis='y', labelcolor='#ff7f0e')
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Combine legends
lines = line1 + line2
if anomaly_mask_p1.any():
    lines = lines + [marker1]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

# Panel 2 - Both scores on same plot
ax2 = axes[1]
ax2_twin = ax2.twinx()

line3 = ax2.plot(df_scores_p2.index, df_scores_p2['anomaly_score'], 
                 alpha=0.7, linewidth=1.5, color='#2ca02c', label='Non-Normalized')
line4 = ax2_twin.plot(df_scores_p2.index, df_scores_p2['anomaly_score_normalized'], 
                      alpha=0.7, linewidth=1.5, color='#d62728', label='Normalized')

# Add anomaly markers
anomaly_mask_p2 = df_scores_p2['anomaly_flag'] == 1
if anomaly_mask_p2.any():
    marker3 = ax2.scatter(df_scores_p2.index[anomaly_mask_p2], df_scores_p2['anomaly_score'][anomaly_mask_p2], 
                          color='red', s=40, marker='o', alpha=0.8, zorder=5, label='Anomaly')
    marker4 = ax2_twin.scatter(df_scores_p2.index[anomaly_mask_p2], df_scores_p2['anomaly_score_normalized'][anomaly_mask_p2], 
                                color='red', s=40, marker='o', alpha=0.8, zorder=5)

ax2.set_title('Panel 2: Normalized vs Non-Normalized Scores', fontsize=12, fontweight='bold')
ax2.set_xlabel('Timestamp')
ax2.set_ylabel('Anomaly Score (Non-Normalized)', color='#2ca02c')
ax2_twin.set_ylabel('Anomaly Score (Normalized)', color='#d62728')
ax2.tick_params(axis='y', labelcolor='#2ca02c')
ax2_twin.tick_params(axis='y', labelcolor='#d62728')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# Combine legends
lines = line3 + line4
if anomaly_mask_p2.any():
    lines = lines + [marker3]
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, loc='upper left')

plt.tight_layout()
plt.show()

## 5. Interactive Plot - Plotly

In [ ]:
# Create interactive plot with Plotly
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Panel 1: Non-Normalized Scores',
        'Panel 1: Normalized Scores [0, 1]',
        'Panel 2: Non-Normalized Scores',
        'Panel 2: Normalized Scores [0, 1]'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.1
)

# Panel 1 - Non-normalized
fig.add_trace(
    go.Scatter(
        x=df_scores_p1.index,
        y=df_scores_p1['anomaly_score'],
        mode='lines',
        name='Panel 1 Non-Normalized',
        line=dict(color='#1f77b4', width=1.5),
        hovertemplate='<b>Panel 1</b><br>Time: %{x}<br>Score: %{y:.4f}<extra></extra>'
    ),
    row=1, col=1
)

# Panel 1 - Anomaly markers (non-normalized)
anomaly_mask_p1 = df_scores_p1['anomaly_flag'] == 1
if anomaly_mask_p1.any():
    fig.add_trace(
        go.Scatter(
            x=df_scores_p1.index[anomaly_mask_p1],
            y=df_scores_p1['anomaly_score'][anomaly_mask_p1],
            mode='markers',
            name='Panel 1 Anomaly',
            marker=dict(color='red', size=8, symbol='circle', opacity=0.8),
            hovertemplate='<b>Anomaly</b><br>Time: %{x}<br>Score: %{y:.4f}<extra></extra>',
            showlegend=True
        ),
        row=1, col=1
    )

# Panel 1 - Normalized
fig.add_trace(
    go.Scatter(
        x=df_scores_p1.index,
        y=df_scores_p1['anomaly_score_normalized'],
        mode='lines',
        name='Panel 1 Normalized',
        line=dict(color='#ff7f0e', width=1.5),
        hovertemplate='<b>Panel 1</b><br>Time: %{x}<br>Normalized Score: %{y:.4f}<extra></extra>'
    ),
    row=1, col=2
)

# Panel 1 - Anomaly markers (normalized)
if anomaly_mask_p1.any():
    fig.add_trace(
        go.Scatter(
            x=df_scores_p1.index[anomaly_mask_p1],
            y=df_scores_p1['anomaly_score_normalized'][anomaly_mask_p1],
            mode='markers',
            name='Panel 1 Anomaly (Normalized)',
            marker=dict(color='red', size=8, symbol='circle', opacity=0.8),
            hovertemplate='<b>Anomaly</b><br>Time: %{x}<br>Normalized Score: %{y:.4f}<extra></extra>',
            showlegend=False
        ),
        row=1, col=2
    )

# Panel 2 - Non-normalized
fig.add_trace(
    go.Scatter(
        x=df_scores_p2.index,
        y=df_scores_p2['anomaly_score'],
        mode='lines',
        name='Panel 2 Non-Normalized',
        line=dict(color='#2ca02c', width=1.5),
        hovertemplate='<b>Panel 2</b><br>Time: %{x}<br>Score: %{y:.4f}<extra></extra>'
    ),
    row=2, col=1
)

# Panel 2 - Anomaly markers (non-normalized)
anomaly_mask_p2 = df_scores_p2['anomaly_flag'] == 1
if anomaly_mask_p2.any():
    fig.add_trace(
        go.Scatter(
            x=df_scores_p2.index[anomaly_mask_p2],
            y=df_scores_p2['anomaly_score'][anomaly_mask_p2],
            mode='markers',
            name='Panel 2 Anomaly',
            marker=dict(color='red', size=8, symbol='circle', opacity=0.8),
            hovertemplate='<b>Anomaly</b><br>Time: %{x}<br>Score: %{y:.4f}<extra></extra>',
            showlegend=True
        ),
        row=2, col=1
    )

# Panel 2 - Normalized
fig.add_trace(
    go.Scatter(
        x=df_scores_p2.index,
        y=df_scores_p2['anomaly_score_normalized'],
        mode='lines',
        name='Panel 2 Normalized',
        line=dict(color='#d62728', width=1.5),
        hovertemplate='<b>Panel 2</b><br>Time: %{x}<br>Normalized Score: %{y:.4f}<extra></extra>'
    ),
    row=2, col=2
)

# Panel 2 - Anomaly markers (normalized)
if anomaly_mask_p2.any():
    fig.add_trace(
        go.Scatter(
            x=df_scores_p2.index[anomaly_mask_p2],
            y=df_scores_p2['anomaly_score_normalized'][anomaly_mask_p2],
            mode='markers',
            name='Panel 2 Anomaly (Normalized)',
            marker=dict(color='red', size=8, symbol='circle', opacity=0.8),
            hovertemplate='<b>Anomaly</b><br>Time: %{x}<br>Normalized Score: %{y:.4f}<extra></extra>',
            showlegend=False
        ),
        row=2, col=2
    )

# Update layout
fig.update_layout(
    title_text='Anomaly Scores: Normalized vs Non-Normalized Comparison',
    title_x=0.5,
    height=800,
    showlegend=True,
    hovermode='x unified'
)

# Update y-axes
fig.update_yaxes(title_text='Anomaly Score', row=1, col=1)
fig.update_yaxes(title_text='Normalized Score [0, 1]', range=[0, 1], row=1, col=2)
fig.update_yaxes(title_text='Anomaly Score', row=2, col=1)
fig.update_yaxes(title_text='Normalized Score [0, 1]', range=[0, 1], row=2, col=2)

# Update x-axes
fig.update_xaxes(title_text='Timestamp', row=2, col=1)
fig.update_xaxes(title_text='Timestamp', row=2, col=2)

fig.show()

## 6. Distribution Comparison

In [ ]:
# Create histogram comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Anomaly Scores Distribution Comparison', fontsize=16, fontweight='bold')

# Panel 1 - Non-normalized histogram
ax1 = axes[0, 0]
ax1.hist(df_scores_p1['anomaly_score'], bins=50, alpha=0.7, color='#1f77b4', edgecolor='black')
ax1.set_title('Panel 1: Non-Normalized Score Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Anomaly Score')
ax1.set_ylabel('Frequency')
ax1.grid(True, alpha=0.3)

# Panel 1 - Normalized histogram
ax2 = axes[0, 1]
ax2.hist(df_scores_p1['anomaly_score_normalized'], bins=50, alpha=0.7, color='#ff7f0e', edgecolor='black')
ax2.set_title('Panel 1: Normalized Score Distribution', fontsize=12, fontweight='bold')
ax2.set_xlabel('Normalized Anomaly Score')
ax2.set_ylabel('Frequency')
ax2.set_xlim([0, 1])
ax2.grid(True, alpha=0.3)

# Panel 2 - Non-normalized histogram
ax3 = axes[1, 0]
ax3.hist(df_scores_p2['anomaly_score'], bins=50, alpha=0.7, color='#2ca02c', edgecolor='black')
ax3.set_title('Panel 2: Non-Normalized Score Distribution', fontsize=12, fontweight='bold')
ax3.set_xlabel('Anomaly Score')
ax3.set_ylabel('Frequency')
ax3.grid(True, alpha=0.3)

# Panel 2 - Normalized histogram
ax4 = axes[1, 1]
ax4.hist(df_scores_p2['anomaly_score_normalized'], bins=50, alpha=0.7, color='#d62728', edgecolor='black')
ax4.set_title('Panel 2: Normalized Score Distribution', fontsize=12, fontweight='bold')
ax4.set_xlabel('Normalized Anomaly Score')
ax4.set_ylabel('Frequency')
ax4.set_xlim([0, 1])
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
# Print detailed statistics
print('=' * 80)
print('ANOMALY SCORES SUMMARY STATISTICS')
print('=' * 80)

print('\n--- Panel 1 ---')
print('\nNon-Normalized Scores:')
print(f'  Min: {df_scores_p1["anomaly_score"].min():.6f}')
print(f'  Max: {df_scores_p1["anomaly_score"].max():.6f}')
print(f'  Mean: {df_scores_p1["anomaly_score"].mean():.6f}')
print(f'  Median: {df_scores_p1["anomaly_score"].median():.6f}')
print(f'  Std: {df_scores_p1["anomaly_score"].std():.6f}')
print(f'  25th percentile: {df_scores_p1["anomaly_score"].quantile(0.25):.6f}')
print(f'  75th percentile: {df_scores_p1["anomaly_score"].quantile(0.75):.6f}')
print(f'  95th percentile: {df_scores_p1["anomaly_score"].quantile(0.95):.6f}')
print(f'  99th percentile: {df_scores_p1["anomaly_score"].quantile(0.99):.6f}')

print('\nNormalized Scores:')
print(f'  Min: {df_scores_p1["anomaly_score_normalized"].min():.6f}')
print(f'  Max: {df_scores_p1["anomaly_score_normalized"].max():.6f}')
print(f'  Mean: {df_scores_p1["anomaly_score_normalized"].mean():.6f}')
print(f'  Median: {df_scores_p1["anomaly_score_normalized"].median():.6f}')
print(f'  Std: {df_scores_p1["anomaly_score_normalized"].std():.6f}')
print(f'  25th percentile: {df_scores_p1["anomaly_score_normalized"].quantile(0.25):.6f}')
print(f'  75th percentile: {df_scores_p1["anomaly_score_normalized"].quantile(0.75):.6f}')
print(f'  95th percentile: {df_scores_p1["anomaly_score_normalized"].quantile(0.95):.6f}')
print(f'  99th percentile: {df_scores_p1["anomaly_score_normalized"].quantile(0.99):.6f}')

print('\n--- Panel 2 ---')
print('\nNon-Normalized Scores:')
print(f'  Min: {df_scores_p2["anomaly_score"].min():.6f}')
print(f'  Max: {df_scores_p2["anomaly_score"].max():.6f}')
print(f'  Mean: {df_scores_p2["anomaly_score"].mean():.6f}')
print(f'  Median: {df_scores_p2["anomaly_score"].median():.6f}')
print(f'  Std: {df_scores_p2["anomaly_score"].std():.6f}')
print(f'  25th percentile: {df_scores_p2["anomaly_score"].quantile(0.25):.6f}')
print(f'  75th percentile: {df_scores_p2["anomaly_score"].quantile(0.75):.6f}')
print(f'  95th percentile: {df_scores_p2["anomaly_score"].quantile(0.95):.6f}')
print(f'  99th percentile: {df_scores_p2["anomaly_score"].quantile(0.99):.6f}')

print('\nNormalized Scores:')
print(f'  Min: {df_scores_p2["anomaly_score_normalized"].min():.6f}')
print(f'  Max: {df_scores_p2["anomaly_score_normalized"].max():.6f}')
print(f'  Mean: {df_scores_p2["anomaly_score_normalized"].mean():.6f}')
print(f'  Median: {df_scores_p2["anomaly_score_normalized"].median():.6f}')
print(f'  Std: {df_scores_p2["anomaly_score_normalized"].std():.6f}')
print(f'  25th percentile: {df_scores_p2["anomaly_score_normalized"].quantile(0.25):.6f}')
print(f'  75th percentile: {df_scores_p2["anomaly_score_normalized"].quantile(0.75):.6f}')
print(f'  95th percentile: {df_scores_p2["anomaly_score_normalized"].quantile(0.95):.6f}')
print(f'  99th percentile: {df_scores_p2["anomaly_score_normalized"].quantile(0.99):.6f}')